# RiskHalo - RAGAS Evaluation Notebook

This notebook focuses on **end-to-end RAGAS evaluation** of the RiskHalo RAG system.

It reuses the existing Python modules under `evaluation/` and `rag/`:
- `evaluation.generate_testset.RiskHaloTestsetGenerator` – builds a persona-driven testset from Chroma session summaries.
- `evaluation.ragas_eval.RiskHaloRagasEvaluator` – runs the full RAG + RAGAS metric pipeline.

Run the cells in order to:
1. Configure the Python path and environment.
2. Verify that session summaries exist in ChromaDB.
3. Generate a persona-based testset.
4. Build the RAGAS-ready dataset (questions, retrieved contexts, answers, ground truth).
5. Execute the full RAGAS evaluation and inspect the metrics.

**Prerequisites:**
- Sessions already embedded into ChromaDB (`riskhalo_sessions` collection populated by your main pipeline / main notebook Step 6).
- `.env` with a valid `OPENAI_API_KEY` (for LLM + embeddings) and any other keys used in `rag/retriever.py`.
- Required Python dependencies installed (`ragas`, `chromadb`, `langchain_openai`, `datasets`, etc.).


In [1]:
import os
import sys
import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed; skipping .env loading.")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

# Ensure project root (where `app/`, `rag/`, `evaluation/` live) is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "RiskHalo_Certification_Challenge"))
if os.path.isdir(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: d:\SOFTWARES\CodeRepo\AIE9_CERTIFICATION_CHALLENGE\RiskHalo_Certification_Challenge\notebooks\RiskHalo_Certification_Challenge


In [2]:
import os, sys
import chromadb  # <-- direct use, no RiskHaloVectorStore import

# True project root = parent of the notebooks directory
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)

# Point to the same ChromaDB directory
persist_dir = os.path.join(PROJECT_ROOT, "chroma_db")

# Use the SAME style of client as RiskHaloTestsetGenerator (no custom Settings)
client = chromadb.PersistentClient(path=persist_dir)
collection = client.get_or_create_collection("riskhalo_sessions")

results = collection.get()
documents = results.get("documents", []) if results else []
print(f"Number of documents retrieved from ChromaDB: {len(documents)}")

PROJECT_ROOT: d:\SOFTWARES\CodeRepo\AIE9_CERTIFICATION_CHALLENGE\RiskHalo_Certification_Challenge
Number of documents retrieved from ChromaDB: 21


In [3]:
from evaluation.generate_testset import RiskHaloTestsetGenerator
import os

# Make sure this matches the earlier verification cell
persist_dir = os.path.join(PROJECT_ROOT, "chroma_db")

# Explicitly point to same ChromaDB and collection
testset_gen = RiskHaloTestsetGenerator(
    collection_name="riskhalo_sessions",
    persist_directory=persist_dir,
)

raw_doc = testset_gen.generate_raw_document()
print("Raw document length:", len(raw_doc))

dataset = testset_gen.generate_dataset()
print("Testset size:", len(dataset))

try:
    display(dataset.to_pandas().head())
except NameError:
    print(dataset.to_pandas().head())

raw_doc ========= RISKHALO HISTORICAL PERFORMANCE RECORD
This document contains concatenated weekly behavioral performance summaries generated by the RiskHalo system.

Each section represents one completed weekly trading analysis.
Sessions are ordered chronologically.
Metrics include behavioral classification, severity score, and overall discipline score.

WEEK 1
Session ID: session_5789bbece1b3
Behavioral State: LOSS_ESCALATION
Severity Score: 1
Discipline Score: N/A

In this session of 7 trades, you were classified as LOSS_ESCALATION (severity 1).

Losses expand following losing trades, indicating emotional risk escalation. Reducing downside variance after drawdowns should be a priority.

Performance Impact
In stable conditions, your trades averaged ₹1080 per trade.
After a loss, performance declined to an average loss of ₹1740 per trade.
This deterioration reflects post-loss loss escalation behavior.
Across the analyzed period, this behavioral shift reduced performance by approximat

,question,context
0,Why do my losses increase after a losing trade?,RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...
1,Am I escalating risk after losses?,RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...
2,Why is my expectancy negative despite winning ...,RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...
3,Am I cutting profits too early?,RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...
4,I follow rules but still lose. Why?,RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...


In [9]:
from datasets import Dataset
from evaluation.generate_testset import RiskHaloTestsetGenerator
from rag.retriever import RiskHaloRetriever

from ragas.metrics import (
    LLMContextRecall,
    ContextEntityRecall,
    ContextPrecision,
    Faithfulness,
    ResponseRelevancy,
)
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

# 1) Use the SAME Chroma path as earlier cells
persist_dir = os.path.join(PROJECT_ROOT, "chroma_db")

# 2) Build the base testset from Chroma using the explicit path
testset_gen = RiskHaloTestsetGenerator(
    collection_name="riskhalo_sessions",
    persist_directory=persist_dir,
)
base_ds = testset_gen.generate_dataset()

print("Base testset Preview:")
print(base_ds.to_pandas())

# 3) Use a retriever that also points to the SAME Chroma path
retriever = RiskHaloRetriever(
    collection_name="riskhalo_sessions",
    persist_directory=persist_dir,
)

ragas_rows = []
for row in base_ds:
    question = row["question"]
    gt_context = row["context"]

    rag_result = retriever.generate(question)
    answer = rag_result["answer"]
    retrieved_contexts = rag_result["retrieved_contexts"]

    ragas_rows.append(
        {
            "question": question,
            "contexts": retrieved_contexts,
            "ground_truth": gt_context,
            "response": answer,
        }
    )

ragas_data = Dataset.from_list(ragas_rows)

print("\nRAGAS dataset :: Preview")
print(ragas_data.to_pandas()[["question", "contexts", "ground_truth", "response"]])

# 4) Run RAGAS evaluation with a stable LLM wrapper
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0)) #gpt-4.1-mini
custom_run_config = RunConfig(timeout=360)

print(f"\nStart RAGAS evaluation")
print(f"evaluate::ragas_data length: {len(ragas_data)}")

results = evaluate(
    ragas_data,
    metrics=[
        LLMContextRecall(),
        ContextEntityRecall(),
        ContextPrecision(),
        Faithfulness(),
        ResponseRelevancy(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

print("\nRAGAS Evaluation Baseline Results - Overall metrics (mean)")
print(results)

# per-question metrics
df = results.to_pandas()
cols = [
    "user_input",
    "context_recall",
    "context_entity_recall",
    "context_precision",
    "faithfulness",
    "answer_relevancy",
]
print("\nRAGAS Evaluation Baseline Results - per-question metrics")
print(df[cols])

raw_doc ========= RISKHALO HISTORICAL PERFORMANCE RECORD
This document contains concatenated weekly behavioral performance summaries generated by the RiskHalo system.

Each section represents one completed weekly trading analysis.
Sessions are ordered chronologically.
Metrics include behavioral classification, severity score, and overall discipline score.

WEEK 1
Session ID: session_5789bbece1b3
Behavioral State: LOSS_ESCALATION
Severity Score: 1
Discipline Score: N/A

In this session of 7 trades, you were classified as LOSS_ESCALATION (severity 1).

Losses expand following losing trades, indicating emotional risk escalation. Reducing downside variance after drawdowns should be a priority.

Performance Impact
In stable conditions, your trades averaged ₹1080 per trade.
After a loss, performance declined to an average loss of ₹1740 per trade.
This deterioration reflects post-loss loss escalation behavior.
Across the analyzed period, this behavioral shift reduced performance by approximat

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Exception raised in Job[45]: TimeoutError()



RAGAS Evaluation Baseline Results - Overall metrics (mean)
{'context_recall': 0.3340, 'context_entity_recall': 0.0854, 'context_precision': 0.8556, 'faithfulness': 0.8565, 'answer_relevancy': 0.8044}

RAGAS Evaluation Baseline Results - per-question metrics
                                          user_input  context_recall  \
0    Why do my losses increase after a losing trade?        0.285714   
1                 Am I escalating risk after losses?        0.333333   
2  Why is my expectancy negative despite winning ...        0.000000   
3                    Am I cutting profits too early?        0.470588   
4                I follow rules but still lose. Why?        0.636364   
5             Is discipline enough to be profitable?        0.000000   
6                          Am I improving over time?        0.387097   
7                   Is my recovery behavior healthy?        0.615385   
8      Is there enough data to evaluate my behavior?        0.277778   
9          What can b

In [8]:
print(df.columns)
print(df.head(2))

Index(['user_input', 'retrieved_contexts', 'response', 'reference',
       'context_recall', 'context_entity_recall', 'context_precision',
       'faithfulness', 'answer_relevancy'],
      dtype='object')
                                        user_input  \
0  Why do my losses increase after a losing trade?   
1               Am I escalating risk after losses?   

                                  retrieved_contexts  \
0  [\nIn this session of 5 trades, you were class...   
1  [\nIn this session of 8 trades, you were class...   

                                            response  \
0  1. Direct Conclusion  \n- Your losses increase...   
1  1. Direct Conclusion  \n- Yes, you are escalat...   

                                           reference  context_recall  \
0  RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...        0.666667   
1  RISKHALO HISTORICAL PERFORMANCE RECORD\nThis d...        0.466667   

   context_entity_recall  context_precision  faithfulness  answer_relevancy  